# 7 — E3: Meta-Circuits — Group Structure

**Hypothesis H4:** Models discover abstract syntactic categories that generalise across languages.

For each hypothesised group of concepts, compute the intersection of their concept-only
neuron sets (the meta-circuit). Test significance via permutation null model.
Compare across languages and models.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["numpy", "pandas", "scipy", "matplotlib", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import ast as ast_mod
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import seaborn as sns
import json

# All (language, model) combos to analyse
COMBOS = [
    {"lang": "P", "model": "QW", "label": "Python/Qwen"},
    # {"lang": "P", "model": "DS", "label": "Python/DeepSeek"},
    {"lang": "R", "model": "QW", "label": "Rust/Qwen"},
    # {"lang": "R", "model": "DS", "label": "Rust/DeepSeek"},
]

# Representative threshold setting
EPSILON = "0.5"
CONSISTENCY = "0.8"
N_LAYERS = 28
REP_LAYER = 14
N_PERMUTATIONS = 1000
SEED = 42

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"



print(f"Combos: {[c['label'] for c in COMBOS]}")
print(f"Settings: eps={EPSILON}, cons={CONSISTENCY}")

In [ ]:
# Cell 2 – Grouping hypotheses

PYTHON_GROUPS = {
    "Iteration":       ["ast__For", "ast__While"],
    "Binding":         ["ast__Assign", "ast__AugAssign", "ast__AnnAssign"],
    "Branching":       ["ast__If", "ast__IfExp"],
    "Function def":    ["ast__FunctionDef", "ast__AsyncFunctionDef", "ast__Lambda"],
    "Error handling":  ["ast__Try", "ast__Raise"],
    "Module import":   ["ast__Import", "ast__ImportFrom"],
    "Comprehensions":  ["ast__ListComp", "ast__SetComp", "ast__DictComp", "ast__GeneratorExp"],
    "Atomicity":       ["ast__Import", "ast__Break", "ast__Pass", "ast__Continue", "ast__Assert"],
}

RUST_GROUPS = {
    "Iteration":       ["rust__For", "rust__Loop", "rust__While"],
    "Binding":         ["rust__Let", "rust__LetMut"],
    "Branching":       ["rust__If", "rust__Match"],
    "Function def":    ["rust__Fn", "rust__Closure"],
    "Error handling":  ["rust__QuestionMark"],
    "Module import":   ["rust__Use", "rust__Mod"],
    "Ownership":       ["rust__Ref", "rust__RefMut", "rust__Deref"],
    "Loop control":    ["rust__Break", "rust__Continue"],
}

LANG_GROUPS = {"P": PYTHON_GROUPS, "R": RUST_GROUPS}

print(f"Python groups: {len(PYTHON_GROUPS)}")
print(f"Rust groups: {len(RUST_GROUPS)}")

In [ ]:
# Cell 3 – Load neuron list CSVs


data = {}  # (lang, model) -> DataFrame

for combo in COMBOS:
    prefix = f"{combo['lang']}_{combo['model']}_"
    fname = f"{prefix}4_neuron_list_eps{EPSILON}_cons{CONSISTENCY}_all_layers_both.csv"
    path = os.path.join(DATA_DIR, fname)

    if not os.path.exists(path):
        # Try split files (half-layer CSVs)
        layers_a = "_".join(str(l) for l in range(14))
        layers_b = "_".join(str(l) for l in range(14, 28))
        pa = os.path.join(DATA_DIR, f"{prefix}4_neuron_list_eps{EPSILON}_cons{CONSISTENCY}_L{layers_a}_both.csv")
        pb = os.path.join(DATA_DIR, f"{prefix}4_neuron_list_eps{EPSILON}_cons{CONSISTENCY}_L{layers_b}_both.csv")
        if os.path.exists(pa) and os.path.exists(pb):
            df = pd.concat([pd.read_csv(pa), pd.read_csv(pb)], ignore_index=True)
        else:
            print(f"  WARNING: missing data for {combo['label']}")
            continue
    else:
        df = pd.read_csv(path)

    key = (combo["lang"], combo["model"])
    data[key] = df
    print(f"  {combo['label']}: {len(df)} rows, {df['object'].nunique()} objects")

print(f"\nLoaded {len(data)} combos")

In [ ]:
# Cell 4 – Helper: parse neuron sets from CSV

def get_concept_only_set(df, obj_name, layer):
    """Get concept-only neuron set for a given object at a given layer."""
    row = df[(df["object"] == obj_name) & (df["layer"] == layer)]
    if len(row) == 0:
        return set()
    indices = ast_mod.literal_eval(row.iloc[0]["concept_only"])
    return set(indices)


def group_intersection(df, members, layer):
    """Compute intersection of concept-only sets for a group at a layer."""
    sets = [get_concept_only_set(df, m, layer) for m in members]
    sets = [s for s in sets if len(s) > 0]  # skip empty
    if len(sets) == 0:
        return set()
    result = sets[0]
    for s in sets[1:]:
        result = result & s
    return result


def group_union(df, members, layer):
    """Compute union of concept-only sets for a group at a layer."""
    result = set()
    for m in members:
        result |= get_concept_only_set(df, m, layer)
    return result


print("Helpers defined.")

In [ ]:
# Cell 5 – Compute meta-circuits + permutation test

rng = np.random.default_rng(SEED)
results = []

for (lang, model), df in data.items():
    groups = LANG_GROUPS.get(lang, {})
    all_objects = sorted(df["object"].unique())

    for group_name, members in groups.items():
        # Filter to members that exist in the data
        available = [m for m in members if m in all_objects]
        if len(available) < 2:
            continue

        for layer in range(N_LAYERS):
            I = group_intersection(df, available, layer)
            U = group_union(df, available, layer)
            sharing = len(I) / len(U) if len(U) > 0 else 0.0

            # Permutation null model
            k = len(available)
            null_sizes = []
            for _ in range(N_PERMUTATIONS):
                random_members = rng.choice(all_objects, size=k, replace=False).tolist()
                null_I = group_intersection(df, random_members, layer)
                null_sizes.append(len(null_I))

            p_value = np.mean([ns >= len(I) for ns in null_sizes]) if len(I) > 0 else 1.0

            results.append({
                "lang": lang, "model": model,
                "group": group_name,
                "layer": layer,
                "n_members": len(available),
                "intersection_size": len(I),
                "union_size": len(U),
                "sharing_ratio": sharing,
                "p_value": p_value,
                "null_mean": np.mean(null_sizes),
            })

    print(f"  {lang}_{model}: {len(groups)} groups x {N_LAYERS} layers done")

df_results = pd.DataFrame(results)
print(f"\nTotal: {len(df_results)} rows")

In [ ]:
# Cell 6 – Jaccard heatmap at representative layer

def compute_jaccard_matrix(df, objects, layer):
    """Compute pairwise Jaccard similarity of concept-only sets."""
    n = len(objects)
    mat = np.zeros((n, n))
    sets = [get_concept_only_set(df, o, layer) for o in objects]
    for i in range(n):
        for j in range(i, n):
            inter = len(sets[i] & sets[j])
            union = len(sets[i] | sets[j])
            sim = inter / union if union > 0 else 0.0
            mat[i, j] = sim
            mat[j, i] = sim
    return mat


for (lang, model), df in data.items():
    label = f"{lang}_{model}"
    objects = sorted(df["object"].unique())
    # Limit to objects with non-trivial concept-only sets at REP_LAYER
    objects = [o for o in objects if len(get_concept_only_set(df, o, REP_LAYER)) > 0]

    if len(objects) < 3:
        print(f"  {label}: too few objects with concept-only neurons at L{REP_LAYER}")
        continue

    mat = compute_jaccard_matrix(df, objects, REP_LAYER)
    short_names = [o.split("__")[1] if "__" in o else o for o in objects]

    fig, ax = plt.subplots(figsize=(max(8, len(objects)*0.4), max(6, len(objects)*0.35)))
    sns.heatmap(mat, xticklabels=short_names, yticklabels=short_names,
                cmap="YlOrRd", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"Jaccard similarity of concept-only sets — {label} L{REP_LAYER}")
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, f"7_E3_jaccard_heatmap_{label}.png"), dpi=150)
    plt.show()

    # Dendrogram
    dist = 1 - mat
    np.fill_diagonal(dist, 0)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method="ward")
    fig, ax = plt.subplots(figsize=(max(10, len(objects)*0.3), 5))
    dendrogram(Z, labels=short_names, ax=ax, leaf_rotation=90, leaf_font_size=7)
    ax.set_title(f"Hierarchical clustering — {label} L{REP_LAYER}")
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, f"7_E3_dendrogram_{label}.png"), dpi=150)
    plt.show()

In [ ]:
# Cell 7 – Summary table

if len(df_results) > 0:
    # Best layer per group (highest sharing ratio)
    summary = (df_results.groupby(["lang", "model", "group"])
               .agg(
                   peak_layer=("sharing_ratio", "idxmax"),
                   max_sharing=("sharing_ratio", "max"),
                   mean_sharing=("sharing_ratio", "mean"),
                   n_sig_layers=("p_value", lambda x: (x < 0.05).sum()),
               ).reset_index())
    # Resolve peak_layer from index
    summary["peak_layer"] = summary["peak_layer"].map(lambda idx: df_results.loc[idx, "layer"])

    print("META-CIRCUIT SUMMARY")
    print("=" * 70)
    display(summary)
else:
    print("No results.")

In [ ]:
# Cell 8 – Save

out_path = os.path.join(DATA_DIR, "7_E3_meta_circuit_results.csv")
df_results.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

print("\n7 complete.")